# Lab 5.1 — TDD with an AI Pair

*Chapter 5 — GenAI Across the SDLC · 45 minutes · JupyterLab + pytest + the OpenAI API*

The recommended pattern for AI-assisted coding, end to end: **you** define
what correct means as a failing pytest suite; the model implements until the
suite goes green; then the model refactors while the suite referees. Red →
green → refactor, with the AI doing the typing and the tests doing the
judging.

The spec and the failing suite are provided below and runnable as shipped.
With no API key (or `COURSE_AI_MOCK=1`) the deterministic mock in `course_ai`
generates a known-good implementation, so the whole red → green loop runs
offline too.

## Objectives

By the end of this lab, you will:

- Run the red → green → refactor loop with an AI pair on a real test runner.
- Drive implementation from a human-written spec and failing tests.
- Iterate on test failures by feeding the failure output back to the model.
- Prove the suite has teeth: a deliberate mutation gets caught.

## Setup

- **Tools:** `pytest` (pre-installed on the VM; `pip install pytest` anywhere
  else), plus the OpenAI API through `course_ai.chat()`.
- **Key:** `OPENAI_API_KEY` from the environment / course `.env`, never
  printed. `COURSE_AI_MOCK=1` (or no key) engages the deterministic mock.
- **Workspace:** the notebook creates `tdd_workspace/` next to itself and
  writes the suite and implementation there. It is regenerated on every run —
  safe to delete.
- **Model:** pinned via `OPENAI_MODEL` (default `gpt-4o-mini`).

In [ ]:
import pathlib
import re
import subprocess
import sys

import course_ai
from course_ai import chat

print("mode:", course_ai.mode())

WORKSPACE = pathlib.Path("tdd_workspace")
WORKSPACE.mkdir(exist_ok=True)

def run_tests():
    """Run pytest on the workspace; return (green: bool, output: str)."""
    r = subprocess.run([sys.executable, "-m", "pytest", str(WORKSPACE),
                        "-q", "--tb=line", "-rf"],
                       capture_output=True, text=True, timeout=120)
    return r.returncode == 0, (r.stdout + r.stderr).strip()

def tail(output, n=12):
    """The last n lines — the part of a pytest run you actually read."""
    return "\n".join(output.splitlines()[-n:])

history = []   # (label, green) — the red -> green loop, captured

## Steps

### Step 1 — The spec (5 min)

You define what correct means. Read the spec — everything the AI writes is
judged against it, and every ambiguity in it becomes a bug you get to debug.

In [ ]:
SPEC = """
parse_semver(version: str) -> tuple[int, int, int]

- Accepts "MAJOR.MINOR.PATCH", e.g. "1.2.3" -> (1, 2, 3).
- An optional leading "v" is allowed: "v1.2.3" -> (1, 2, 3).
- Returns a tuple of three ints.
- Raises ValueError for anything else: "1.2", "1.2.x", "", "1.2.3.4", "v".
"""
print(SPEC)

### Step 2 — The failing suite: red first (5 min)

The suite below encodes the spec — provided for you here; on your own project
you write it yourself. Run it **before any implementation exists** and watch
it go red. Skipping the red bar means you never proved the suite can fail —
a suite that cannot fail cannot protect you.

Stretch (optional): add one more test of your own from the spec — append it
to the test file and re-run to see it red first.

In [ ]:
TEST_FILE = '''
import pytest

from semver_util import parse_semver


def test_happy_path():
    assert parse_semver("1.2.3") == (1, 2, 3)


def test_zeros_and_double_digits():
    assert parse_semver("0.0.0") == (0, 0, 0)
    assert parse_semver("10.20.30") == (10, 20, 30)


def test_optional_v_prefix():
    assert parse_semver("v2.0.1") == (2, 0, 1)


@pytest.mark.parametrize("bad", ["1.2", "1.2.x", "", "1.2.3.4", "v"])
def test_malformed_raises(bad):
    with pytest.raises(ValueError):
        parse_semver(bad)


def test_returns_tuple_of_ints():
    result = parse_semver("3.4.5")
    assert isinstance(result, tuple)
    assert all(isinstance(n, int) for n in result)
'''
(WORKSPACE / "test_semver_util.py").write_text(TEST_FILE)

impl = WORKSPACE / "semver_util.py"
if impl.exists():
    impl.unlink()          # start red: no implementation yet

green, out = run_tests()
print(tail(out))
history.append(("no implementation (suite must fail)", green))
print("\nRED as expected?", not green)

# YOUR CODE (stretch): append one more spec-derived test to
# WORKSPACE / "test_semver_util.py" and re-run run_tests() to see it red.

### Step 3 — Generate the implementation with the API (10 min)

Now the pair does the typing. Write a prompt that hands the model the spec
and asks for the complete `semver_util.py` as one code block. The cell
extracts the code, writes the file, and re-runs the suite.

![The agentic TDD loop — in this lab you run the same cycle by hand, with the suite as referee](diagrams/ch05_agentic_tdd_loop.png)

*The agentic TDD loop — in this lab you run the same cycle by hand, with the suite as referee (Chapter 5 deck).*

In [ ]:
impl_reply = None
# YOUR CODE: write the generation prompt and call chat(), e.g.
#   impl_reply = chat("You are my AI pair. Write the complete semver_util.py "
#                     "implementing the spec below as one python code block, "
#                     "no prose. Spec:\n" + SPEC)
if impl_reply is None:
    impl_reply = chat("You are my AI pair. Write the complete semver_util.py "
                      "implementing parse_semver per this spec, as one python "
                      "code block and nothing else:\n" + SPEC)
    print("(mock/deterministic implementation — write your own prompt above to run yours)\n")

m = re.search(r"```(?:python)?\n(.*?)```", impl_reply, re.DOTALL)
code_text = (m.group(1) if m else impl_reply).strip() + "\n"
impl.write_text(code_text)
print("--- generated semver_util.py ---")
print(code_text)

green, out = run_tests()
history.append(("first generated implementation", green))
print(tail(out))
print("\nGREEN?", green)

### Step 4 — Iterate until green (10 min)

If the suite is red, don't touch the implementation by hand yet — feed the
failure output back to the model and let it repair. That *is* the AI-TDD
loop: the tests carry your intent, the model carries the typing. The loop
below stops at three attempts; in mock mode the first implementation is
already green, so read the loop and check the history table.

In [ ]:
MAX_ATTEMPTS = 3
attempt = 1
while not green and attempt < MAX_ATTEMPTS:
    attempt += 1
    repair_reply = chat("The pytest run failed with this output:\n" + tail(out) +
                        "\n\nFix semver_util.py so the whole suite passes. Return the "
                        "complete corrected file as one python code block. Spec:\n" + SPEC)
    m = re.search(r"```(?:python)?\n(.*?)```", repair_reply, re.DOTALL)
    impl.write_text(((m.group(1) if m else repair_reply).strip() + "\n"))
    green, out = run_tests()
    history.append((f"repair attempt {attempt}", green))
    print(f"--- after repair attempt {attempt} ---")
    print(tail(out))

print("\nred -> green history:")
for label, ok in history:
    print(f"  [{'GREEN' if ok else 'RED  '}] {label}")
if not green:
    print("\nStill red after", MAX_ATTEMPTS - 1, "repairs — tighten the spec or the prompt,")
    print("or fix the last line yourself and note what the model kept missing.")

### Step 5 — Refactor with AI, suite as referee (10 min)

Green means you may refactor. Ask the model to refactor **for readability
only** and state the constraint explicitly: behaviour must not change. Then
re-run the suite — if it goes red, the refactor is rejected, no debate. That
is the whole point of doing this inside TDD instead of after it.

In [ ]:
refactor_reply = None
# YOUR CODE: ask the model to refactor the current semver_util.py for
# readability, stating the constraint, e.g.
#   refactor_reply = chat("Refactor this implementation for readability. "
#                         "Behaviour must not change — the test suite is the "
#                         "referee. Return the complete file as one code "
#                         "block:\n" + impl.read_text())
if refactor_reply is None:
    refactor_reply = chat("Refactor this parse_semver implementation for readability. "
                          "Behaviour must not change — the test suite is the referee. "
                          "Return the complete file as one python code block:\n"
                          + impl.read_text())
    print("(mock/deterministic refactor — write your own prompt above to run yours)\n")

m = re.search(r"```(?:python)?\n(.*?)```", refactor_reply, re.DOTALL)
impl.write_text(((m.group(1) if m else refactor_reply).strip() + "\n"))
print("--- refactored semver_util.py ---")
print(impl.read_text())

green, out = run_tests()
history.append(("refactored implementation", green))
print(tail(out))
assert green, "the refactor broke the suite — revert it (git checkout) or repair and re-run"
print("\nStill green after refactor — the suite refereed the change.")

### Step 6 — Stretch: prove the suite has teeth (5 min)

A suite that never fails is a rumor. Introduce a **deliberate mutation** (one
line) and watch a test catch it — this is the seed of mutation testing, and
the reason you can trust the green bar in Steps 3–5. The cell picks a
mutation that matches whichever implementation is currently in the file,
then **restores** the file afterward: never leave a mutation in the tree.

In [ ]:
src = impl.read_text()
MUTATIONS = [
    ("len(parts) != 3", "len(parts) != 2"),          # off-by-one on the shape check
    (r"(\d+)\.(\d+)\.(\d+)", r"(\d+)\.(\d+)"),  # drop the patch component
]
mutated, note = None, None
for old, new in MUTATIONS:
    if old in src:
        mutated, note = src.replace(old, new, 1), f"{old!r} -> {new!r}"
        break
if mutated is None and "return" in src:              # generic fallback for any other shape
    mutated = src.replace("return", "return None and", 1)
    note = "first 'return' -> 'return None and ...'"

if mutated is None:
    print("no applicable mutation found — write one by hand: change one operator and re-run")
else:
    impl.write_text(mutated)
    green_mut, out_mut = run_tests()
    print(f"mutation applied: {note}\n")
    print(tail(out_mut))
    caught = [ln for ln in out_mut.splitlines() if ln.startswith("FAILED")]
    print("\nmutation caught by the suite?", not green_mut)
    if caught:
        print("caught by:")
        for ln in caught:
            print(" ", ln)
    # restore — the suite should go straight back to green
    impl.write_text(src)
    green, out = run_tests()
    history.append(("after restoring the mutation", green))
    print("\nrestored and green?", green)

## Deliverable

1. The red → green history table (Step 4), showing the suite red before the
   implementation existed and green after.
2. The final green run **after** the AI refactor (Step 5's assertion passing
   is your proof).
3. The mutation output (Step 6): which test caught the deliberate bug.

## Reflection

1. What did writing the spec *before* generating code change about the bugs
   you had to fix?
2. When the repair loop fails twice, what is usually wrong — the model, the
   prompt, or the spec?
3. Which step would you skip on a real ticket, and what would it cost you?

## Debrief (instructor-led)

1. Who skipped the red bar in Step 2 and generated first? What couldn't they
   prove afterward?
2. What belongs in the repair prompt: the whole pytest output, the tail, or
   just the failing assertion? Try it and compare repair quality.
3. The mutation step is a manual version of mutation testing. What would it
   take to run mutations automatically in CI (tools like `mutmut` or
   `cosmic-ray`)?
4. Bridge to Chapter 6: a test suite judges code. What judges a *prompt*?

## Troubleshooting

- **`ModuleNotFoundError: semver_util` in the red step** — expected: that IS
  the red bar. Continue to Step 3.
- **`No module named pytest`** — `pip install pytest` (pre-installed on the
  VM). The notebook shells out to `sys.executable -m pytest`, so the package
  must live in the same environment as the kernel.
- **The generated file has prose around the code** — the extractor looks for
  a ```python fence; ask the model for "one code block and nothing else".
  If there is no fence, the whole reply is written to the file, which usually
  fails — let the Step 4 repair loop handle it.
- **Green on the first generation in mock mode** — expected: the mock returns
  a known-good implementation deterministically. The loop and history still
  show the mechanics; with a key, the first attempt often is red.
- **The assertion in Step 5 fires** — the refactor changed behaviour. Revert
  (rewrite the previous file contents) or repair, but do not weaken the tests
  to make it pass.
- **Mutation step prints "no applicable mutation found"** — your live-model
  implementation has an unusual shape. Introduce any one-character bug by
  hand and re-run `run_tests()`.